# Bloomberg Pricing Ingestion

Process priced deals from Bloomberg → update deal/transaction/manager records.

**Workflow:**
1. Drop your `pricings_YYYYMMDD.csv` in the project root
2. Run Cell 1 (setup)
3. Run Cell 2 (preview CSV — see what's priced)
4. Run Cell 3 (run pipeline — xlwings pulls BQL tranche data + updates stores)
5. Run Cell 4 (review what was updated)

**Announcements** come from emails. **Pricings** come from here.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 1: SETUP
# ═══════════════════════════════════════════════════════════════════════

import sys, os, glob
ROOT = os.path.dirname(os.path.abspath('__file__'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from backend.bbg_pricing import (
    parse_pricing_csv, run_pricing_pipeline,
    format_tranche_pricing_from_bql, LEAD_MGR_MAP, COLLATERAL_MAP,
)
from backend import store

# Find the most recent pricing CSV
csvs = sorted(glob.glob(os.path.join(ROOT, 'pricings_*.csv')))
if csvs:
    CSV_PATH = csvs[-1]
    print(f'Found: {os.path.basename(CSV_PATH)}')
else:
    CSV_PATH = None
    print('No pricings_*.csv found in project root.')
    print('Drop your Bloomberg pricing CSV here and re-run.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 2: PREVIEW — See what's in the CSV before processing
# ═══════════════════════════════════════════════════════════════════════

if CSV_PATH:
    deals = parse_pricing_csv(CSV_PATH)
    print(f'{len(deals)} priced deals found\n')
    print(f'{"Deal Name":25s} {"Pricing":12s} {"Settle":12s} {"Size(MM)":>10s} {"Type":8s} {"Lead":15s} {"Txn Type":12s}')
    print('─' * 100)
    for d in deals:
        size = f"{d['orig_mm']:,.0f}" if d['orig_mm'] else '?'
        print(f"{d['deal_name']:25s} {d['pricing_date']:12s} {d['settle_date']:12s} {size:>10s} {d['deal_type']:8s} {d['lead_mgr_full']:15s} {d['transaction_type']:12s}")

    # Check which deals already exist in our stores
    existing_deals = store.read_store(store.DEALS)
    existing_names = {d.get('deal_name', '') for d in existing_deals}
    existing_txns = store.read_store(store.TRANSACTIONS)
    txn_names = {t.get('deal_name', '') for t in existing_txns}

    new_deals = [d for d in deals if d['deal_name'] not in existing_names]
    known_deals = [d for d in deals if d['deal_name'] in existing_names]

    print(f'\n{len(known_deals)} already in deals.json (will update)')
    print(f'{len(new_deals)} new deals (will create)')
    print(f'{len([d for d in deals if d["deal_name"] in txn_names])} have existing transactions (will mark Priced)')
else:
    print('No CSV loaded — run Cell 1 first.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 3: RUN PIPELINE — Process all priced deals
# ═══════════════════════════════════════════════════════════════════════
#   Set use_xlwings=True if Bloomberg Terminal + Excel are running.
#   Set use_xlwings=False to update stores from CSV data only (no
#   tranche-level detail, but still marks deals as Priced).
# ═══════════════════════════════════════════════════════════════════════

USE_XLWINGS = True  # Set to False on machines without Bloomberg/Excel

if CSV_PATH:
    results = run_pricing_pipeline(CSV_PATH, use_xlwings=USE_XLWINGS)
else:
    print('No CSV loaded.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 4: REVIEW — See what was updated
# ═══════════════════════════════════════════════════════════════════════

if 'results' in dir():
    created = [r for r in results if r['summary']['deal'] == 'created']
    updated = [r for r in results if r['summary']['deal'] == 'updated']
    new_mgrs = [r for r in results if r['summary'].get('manager')]

    print(f'RESULTS SUMMARY')
    print(f'═' * 60)
    print(f'  Deals created:     {len(created)}')
    print(f'  Deals updated:     {len(updated)}')
    print(f'  New managers:      {len(new_mgrs)}')
    print(f'  Tranches fetched:  {sum(r["tranches"] for r in results)}')
    print()

    if created:
        print('New deals:')
        for r in created:
            print(f'  + {r["deal_name"]} ({r["deal_type"]}, {r["lead_mgr_full"]}, {r["orig_mm"]:,.0f}MM)')
    if new_mgrs:
        print('New managers:')
        for r in new_mgrs:
            print(f'  + {r["summary"]["manager"]}')

    # Show latest transactions
    print(f'\nRecent Priced transactions:')
    txns = store.read_store(store.TRANSACTIONS)
    priced = [t for t in txns if t.get('status') == 'Priced']
    for t in priced[-10:]:
        fp = (t.get('final_pricing') or '')[:60]
        print(f'  {t["deal_name"]:25s} | {t.get("priced_date","?"):12s} | {fp}')
else:
    print('Run Cell 3 first.')